# Fingerprint benchmark

Source: archived main notebook, cells 62–70. The old code replaced the fingerprint list with a single matrix and fit PCA before cross-validation. This version retains every available representation and fits PCA within each training fold. Fixed hyperparameters provide a transparent baseline, not the archived hyperopt search.


In [ ]:
from pathlib import Path
import os
import sys

def repository_root(start):
    for path in (start, *start.parents):
        if (path / "Code" / "model_train.py").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the CycloAddGenerator repository.")

REPO_ROOT = repository_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks" / "03_modeling"))
DATA_DIR = REPO_ROOT / "Data" / "DFT_Result"
DESCRIPTOR_MAP = REPO_ROOT / "Data" / "Descriptors" / "Descriptors_.pkl"

In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import (AdaBoostRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, RandomForestRegressor)
from sklearn.linear_model import BayesianRidge, Lasso, Ridge, SGDRegressor
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor
from _modeling_common import metrics, new_catboost

bundle_path = DATA_DIR / "Descriptors" / "ZINC_fingerprint_bundle.pkl"
with bundle_path.open("rb") as handle:
    bundle = pickle.load(handle)
fingerprints, targets = bundle["features"], bundle["targets"]
row_count = len(bundle["reaction_keys"])
if any(len(x) != row_count for x in fingerprints.values()):
    raise ValueError("The fingerprint matrices contain different reactions")

In [ ]:
MODEL_FACTORIES = {
    "GB": lambda: GradientBoostingRegressor(random_state=0),
    "RF": lambda: RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1),
    "ADB": lambda: AdaBoostRegressor(n_estimators=200, random_state=0),
    "ET": lambda: ExtraTreesRegressor(n_estimators=200, random_state=0, n_jobs=-1),
    "XGB": lambda: XGBRegressor(n_estimators=200, max_depth=5, random_state=0),
    "Lasso": lambda: Lasso(alpha=0.5), "Ridge": lambda: Ridge(alpha=0.5),
    "BR": lambda: BayesianRidge(), "SGD": lambda: SGDRegressor(random_state=0),
    "SVR": lambda: SVR(C=1.0), "KNN": lambda: KNeighborsRegressor(n_neighbors=5),
    "CatBoost": new_catboost,
}

def estimator_for(name, n_rows, n_features):
    steps = []
    if n_features > 300:
        steps.append(PCA(n_components=min(300, n_rows - 1, n_features),
                         whiten=True, random_state=0))
    if name in {"Lasso", "Ridge", "BR", "SGD", "SVR", "KNN"}:
        steps.append(StandardScaler())
    steps.append(MODEL_FACTORIES[name]())
    return make_pipeline(*steps)

## Five-fold model comparison

Select fewer models or representations before running if only a subset is needed.


In [ ]:
SELECTED_MODELS = list(MODEL_FACTORIES)
SELECTED_REPRESENTATIONS = list(fingerprints)
folds = list(KFold(5, shuffle=True, random_state=0).split(np.arange(row_count)))
rows = []
for representation in SELECTED_REPRESENTATIONS:
    x = np.asarray(fingerprints[representation], dtype=float)
    for target, y in targets.items():
        y = np.asarray(y, dtype=float)
        for name in SELECTED_MODELS:
            for fold, (train_ids, test_ids) in enumerate(folds):
                model = estimator_for(name, len(train_ids), x.shape[1])
                model.fit(x[train_ids], y[train_ids])
                rows.append({"representation": representation, "target": target,
                             "model": name, "fold": fold,
                             **metrics(y[test_ids], model.predict(x[test_ids]))})
benchmark = pd.DataFrame(rows)
display(benchmark.groupby(["target", "representation", "model"])[["r2", "mae"]].mean())